In [1]:
%pip install ray[tune]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.3/72.3 MB 19.1 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1


In [2]:
%pip install lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.0/846.0 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.5/849.5 kB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.4/242.4 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.6/221.6 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.3/377.3 kB 32.8 MB/s eta 0:00:00


In [5]:
import torch
import numpy as np
import random

seed = 365

# 1. Set Python random seed
random.seed(seed)

# 2. Set Numpy seed
np.random.seed(seed)

# 3. Set PyTorch seeds
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# 4. Force deterministic algorithms (slower but reproducible)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Global seed set to: {seed}")

Global seed set to: 365


In [5]:
import torch
import lightning as L
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning as L
from torchmetrics.classification import BinaryAccuracy


In [6]:
L.seed_everything(365, workers = True)

Seed set to 365


365

In [7]:

class DLGN_Module(L.LightningModule):
    def __init__(self, input_dim, hidden_dim, num_layers, beta=1, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.beta = beta
        self.lr = lr
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.hiddens = torch.nn.ParameterList([
            torch.nn.Parameter(torch.randn(input_dim, hidden_dim), requires_grad=True)
        ])

        self.hiddens.extend([
            torch.nn.Parameter(torch.randn(hidden_dim, hidden_dim), requires_grad=True) 
            for _ in range(num_layers - 1)
        ])
        

        self.Us = torch.nn.ParameterList([
            torch.nn.Parameter(torch.randn(hidden_dim, 1), requires_grad=True)
        ])

        self.Us.extend([
            torch.nn.Parameter(torch.randn(hidden_dim, hidden_dim), requires_grad=True) 
            for _ in range(num_layers - 1)
        ])
        # Last U: (hidden, 1) -> This acts as the final output projection
        ulast = torch.nn.Parameter(torch.randn(hidden_dim, 1))
        self.Us.extend([ulast])
        
        # 3. Biases
        self.biases = torch.nn.ParameterList([
            torch.nn.Parameter(torch.randn(hidden_dim)) for _ in range(num_layers)
        ])


        self.criterion = nn.BCEWithLogitsLoss()
        self.train_acc = BinaryAccuracy()
        self.val_acc = BinaryAccuracy()

    def forward(self, x):
        curr_input = x


        for i in range(self.num_layers):
            W = self.hiddens[i]
            b = self.biases[i]
            U = self.Us[i]


            linear_out = torch.matmul(curr_input, W) + b
            

            gate_score = torch.matmul(linear_out, U)
            

            gate = torch.sigmoid(self.beta * gate_score)
            

            curr_input = linear_out * gate

        final_u = self.Us[-1]
        out = torch.matmul(curr_input, final_u)
        
        return out

    def training_step(self, batch, batch_idx):
        x, y = batch
        

        logits = self(x)
        logits = logits.squeeze(1) 
        
    
        y = y.float()
        
        loss = self.criterion(logits, y)

        preds = torch.sigmoid(logits)
        acc = self.train_acc(preds, y)
   
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("train_acc", acc, on_step=False, on_epoch=True, prog_bar=True)
        
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        logits = logits.squeeze(1)
        y = y.float()
        
        loss = self.criterion(logits, y)
        preds = torch.sigmoid(logits)
        acc = self.val_acc(preds, y)
        
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)
        

In [8]:
from torch.utils.data import TensorDataset, DataLoader, random_split
import lightning as L
import numpy as np

class NumpyDataModule(L.LightningDataModule):
    def __init__(self, X, y, val_split=0.2, batch_size=32):
        super().__init__()
        self.X = X
        self.y = y
        self.val_split = val_split
        self.batch_size = batch_size
        
        self.train_data = None
        self.val_data = None

    def setup(self, stage=None):
        full_dataset = TensorDataset(
            torch.from_numpy(self.X).float(), 
            torch.from_numpy(self.y).float()
        )
        

        val_size = int(len(full_dataset) * self.val_split)
        train_size = len(full_dataset) - val_size
        

        self.train_data, self.val_data = random_split(
            full_dataset, 
            [train_size, val_size]
        )

    def train_dataloader(self):
        return DataLoader(self.train_data, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_data, batch_size=self.batch_size, shuffle=False)

In [6]:
import torch
import lightning as L

class DLGN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, beta=1):
        super(DLGN, self).__init__()
        self.beta = beta
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.hiddens = torch.nn.ParameterList([torch.nn.Parameter(torch.randn(input_dim, hidden_dim), requires_grad=True)])
        self.hiddens.extend([torch.nn.Parameter(torch.randn(hidden_dim, hidden_dim), requires_grad=True) for _ in range(num_layers - 1)])
        
        self.Us = torch.nn.ParameterList([torch.nn.Parameter(torch.randn(hidden_dim, 1), requires_grad=True)])
        self.Us.extend([torch.nn.Parameter(torch.randn(hidden_dim, hidden_dim), requires_grad=True) for _ in range(num_layers - 1)])
        ulast = torch.nn.Parameter(torch.randn(hidden_dim, 1))
        self.Us.extend([ulast])
        
        self.biases = torch.nn.ParameterList([torch.nn.Parameter(torch.randn(hidden_dim)) for _ in range(num_layers)])

    def forward(self, x):
        self.V = [self.hiddens[0]]
        h = torch.sigmoid(self.beta * (x @ self.V[0] + self.biases[0])) * self.Us[0].T

        for i in range(1, self.num_layers):
            self.V.append(self.V[-1] @ self.hiddens[i])
            h = torch.sigmoid(self.beta * (x @ self.V[i] + self.biases[i])) * (h @ self.Us[i])
        
        return h @ self.Us[-1]

# class DLGN_Module(L.LightningModule):
#     def __init__(self, input_dim, hidden_dim, num_layers, beta=1):
#         super().__init__()
#         self.beta = beta
#         self.input_dim = input_dim
#         self.hidden_dim = hidden_dim
#         self.num_layers = num_layers

#         self.hiddens = torch.nn.ParameterList([torch.nn.Parameter(torch.randn(input_dim, hidden_dim), requires_grad=True)])
#         self.hiddens.extend([torch.nn.Parameter(torch.randn(hidden_dim, hidden_dim), requires_grad=True) for _ in range(num_layers - 1)])
        
#         self.Us = torch.nn.ParameterList([torch.nn.Parameter(torch.randn(hidden_dim, 1), requires_grad=True)])
#         self.Us.extend([torch.nn.Parameter(torch.randn(hidden_dim, hidden_dim), requires_grad=True) for _ in range(num_layers - 1)])
#         ulast = torch.nn.Parameter(torch.randn(hidden_dim, 1))
#         self.Us.extend([ulast])
        
#         self.biases = torch.nn.ParameterList([torch.nn.Parameter(torch.randn(hidden_dim)) for _ in range(num_layers)])


        


In [1]:

def data_gen_decision_tree(num_data=1000, dim=2, seed=0, w_list=None, b_list=None,vals=None, num_levels=2, threshold = 0):        
    #set_npseed(seed=seed)
    import numpy as np
    np.random.seed(seed)
    

    # Construct a complete decision tree with 2**num_levels-1 internal nodes,
    # e.g. num_levels=2 means there are 3 internal nodes.
    # w_list, b_list is a list of size equal to num_internal_nodes
    # vals is a list of size equal to num_leaf_nodes, with values +1 or 0
    num_internal_nodes = 2**num_levels - 1
    num_leaf_nodes = 2**num_levels
    stats = np.zeros(num_internal_nodes+num_leaf_nodes) #stores the num of datapoints at each node so at 0(root) all data points will be present

    if vals is None: #when val i.e., labels are not provided make the labels dynamically
        vals = np.arange(0,num_internal_nodes+num_leaf_nodes,1,dtype=np.int32)%2 #assign 0 or 1 label to the node based on whether its numbering is even or odd
        vals[:num_internal_nodes] = -99 #we put -99 to the internal nodes as only the values of leaf nodes are counted

    if w_list is None: #if the w values of the nodes (hyperplane eqn) are not provided then generate dynamically
        w_list = np.random.standard_normal((num_internal_nodes, dim))
        w_list = w_list/np.linalg.norm(w_list, axis=1)[:, None] #unit norm w vects
        b_list = np.zeros((num_internal_nodes))

    

#     data_x = np.random.random_sample((num_data, dim))*2 - 1. #generate the datas in range -1 to +1
#     relevant_stats = data_x @ w_list.T + b_list #stores the x.wT+b value of each nodes for all data points(num_data x num_nodes) to check if > 0 i.e will follow right sub tree route or <0 and will follow left sub tree route
#     curr_index = np.zeros(shape=(num_data), dtype=int) #stores the curr index for each data point from root to leaf. So initially a datapoint starts from root but then it can go to right or left if it goes to right its curr index will become 2 from 0 else 1 from 0 then in next iteration from say 2 it goes to right then it will become 6

    data_x = np.random.standard_normal((num_data, dim))
    data_x /= np.sqrt(np.sum(data_x**2, axis=1, keepdims=True))
    relevant_stats = data_x @ w_list.T + b_list
    curr_index = np.zeros(shape=(num_data), dtype=int)
    
    for level in range(num_levels):
        nodes_curr_level=list(range(2**level - 1,2**(level+1)-1  ))
        for el in nodes_curr_level:
#             b_list[el]=-1*np.median(relevant_stats[curr_index==el,el])
            relevant_stats[:,el] += b_list[el]
        decision_variable = np.choose(curr_index, relevant_stats.T) #based on the curr index will choose the corresponding node value of the datapoint

        # Go down and right if wx+b>0 down and left otherwise.
        # i.e. 0 -> 1 if w[0]x+b[0]<0 and 0->2 otherwise
        curr_index = (curr_index+1)*2 - (1-(decision_variable > 0)) #update curr index based on the desc_variable
        

    bound_dist = np.min(np.abs(relevant_stats), axis=1) #finds the abs value of the minm node value of a datapoint. If some node value of a datapoint is 0 then that data point exactly passes through a hyperplane and we remove all such datapoints
    thres = threshold
    labels = vals[curr_index] #finally labels for each datapoint is assigned after traversing the whole tree

    data_x_pruned = data_x[bound_dist>thres] #to distingush the hyperplanes seperately for 0 1 labels (classification)
    #removes all the datapoints that passes through a node hyperplane
    labels_pruned = labels[bound_dist>thres]
    relevant_stats = np.sign(data_x_pruned @ w_list.T + b_list) #storing only +1 or -1 for a particular node if it is active or not
    nodes_active = np.zeros((len(data_x_pruned),  num_internal_nodes+num_leaf_nodes), dtype=np.int32) #stores node actv or not for a data

    for node in range(num_internal_nodes+num_leaf_nodes):
        if node==0:
            stats[node]=len(relevant_stats) #for root node all datapoints are present
            nodes_active[:,0]=1 #root node all data points active status is +1
            continue
        parent = (node-1)//2
        nodes_active[:,node]=nodes_active[:,parent]
        right_child = node-(parent*2)-1 # 0 means left, 1 means right 1 has children 3,4
        #finds if it is a right child or left of the parent
        if right_child==1:
            nodes_active[:,node] *= relevant_stats[:,parent]>0 #if parent node val was >0 then this right child of parent is active
        if right_child==0:
            nodes_active[:,node] *= relevant_stats[:,parent]<0 #else left is active
        stats = nodes_active.sum(axis=0) #updates the status i.e., no of datapoints active in that node (root has all active then gradually divided in left right)
    return ((data_x_pruned, labels_pruned), (w_list, b_list, vals), stats)

In [2]:
import numpy as np
_data,params,stats = data_gen_decision_tree(num_data=40000, dim = 20, seed = 365,num_levels=4)

In [3]:
data = (_data[0].astype(np.float32),_data[1].astype(np.float32))

In [4]:
X,y = data

In [ ]:

L.seed_everything(365)
INPUT_DIM = X.shape[1]


dm = NumpyDataModule(X, y, val_split=0.2, batch_size=32)


model = DLGN_Module(
    input_dim=INPUT_DIM,
    num_layers=20,    
    hidden_dim=40,    
    
)


trainer = L.Trainer(
    max_epochs=20, 
    accelerator="auto",  
    devices=1,
    enable_progress_bar=True
)

# --- 5. Fit ---
trainer.fit(model, datamodule=dm)

Seed set to 365
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ hiddens   │ ParameterList     │ 31.2 K │ train │     0 │
│ 1 │ Us        │ ParameterList     │ 30.5 K │ train │     0 │
│ 2 │ biases    │ ParameterList     │    800 │ train │     0 │
│ 3 │ criterion │ BCEWithLogitsLoss │      0 │ train │     0 │
│ 4 │ train_acc │ BinaryAccuracy    │      0 │ train │     0 │
│ 5 │ val_acc   │ BinaryAccuracy    │      0 │ train │     0 │
└───┴───────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 62.5 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 62.5 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 6                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()


Detected KeyboardInterrupt, attempting graceful shutdown ...


AttributeError: 'tuple' object has no attribute 'tb_frame'

In [13]:
from torch.utils.data import TensorDataset, DataLoader, random_split

tensor_x = torch.from_numpy(X).float()
tensor_y = torch.from_numpy(y).float().unsqueeze(1) 

dataset = TensorDataset(tensor_x, tensor_y)

val_ratio = 0.5
val_size = int(len(dataset) * val_ratio)
train_size = len(dataset) - val_size


train_ds, val_ds = random_split(dataset, [train_size, val_size])


BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")

Train samples: 20000 | Val samples: 20000


In [14]:
import torch.nn as nn
import torch.optim as optim


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")


model = DLGN(input_dim=X.shape[1], hidden_dim=200, num_layers=4, beta=10)
model.to(device)


optimizer = optim.Adam(model.parameters(), lr=0.001,weight_decay=0.1)
criterion = nn.BCEWithLogitsLoss() 


EPOCHS = 100

for epoch in range(EPOCHS):

    model.train()
    train_loss = 0.0
    
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        
 
        optimizer.zero_grad()
        
   
        outputs = model(inputs)
        loss = criterion(outputs, targets)
  
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)


    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item()
            

            predicted = torch.sigmoid(outputs) > 0.5
            total += targets.size(0)
            correct += (predicted == targets).sum().item()
            
    avg_val_loss = val_loss / len(val_loader)
    val_acc = correct / total if total > 0 else 0
    

    print(f"Epoch {epoch+1:02d} | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"Val Acc: {val_acc:.4f}")

Training on: cuda
Epoch 01 | Train Loss: 1839.1154 | Val Loss: 1151.4982 | Val Acc: 0.5678
Epoch 02 | Train Loss: 833.1926 | Val Loss: 730.0170 | Val Acc: 0.5868
Epoch 03 | Train Loss: 535.6878 | Val Loss: 525.3329 | Val Acc: 0.5854
Epoch 04 | Train Loss: 369.4828 | Val Loss: 397.0755 | Val Acc: 0.5869
Epoch 05 | Train Loss: 263.4066 | Val Loss: 295.7440 | Val Acc: 0.5965
Epoch 06 | Train Loss: 190.6904 | Val Loss: 222.5777 | Val Acc: 0.6017
Epoch 07 | Train Loss: 131.9229 | Val Loss: 162.9934 | Val Acc: 0.6010
Epoch 08 | Train Loss: 92.8006 | Val Loss: 117.4240 | Val Acc: 0.6139
Epoch 09 | Train Loss: 61.4572 | Val Loss: 83.2801 | Val Acc: 0.6126
Epoch 10 | Train Loss: 42.3591 | Val Loss: 57.2979 | Val Acc: 0.6088
Epoch 11 | Train Loss: 27.2451 | Val Loss: 36.5491 | Val Acc: 0.6125
Epoch 12 | Train Loss: 15.9946 | Val Loss: 21.8239 | Val Acc: 0.6175
Epoch 13 | Train Loss: 8.9143 | Val Loss: 12.3761 | Val Acc: 0.6122
Epoch 14 | Train Loss: 4.5697 | Val Loss: 6.2228 | Val Acc: 0.6143
Ep

KeyboardInterrupt: 

In [10]:
torch.save(model.state_dict(), 'dlgn_model_h_64_n_4_b_1.pth')

print("Model saved successfully to 'dlgn_model.pth'")

Model saved successfully to 'dlgn_model.pth'
